# Robyn Preparation

Prepare data for Robyn Marketing Mix Modeling.

In [2]:
from pathlib import Path
import pandas as pd

# Input/output paths
input_path = Path('../data/processed/feature_engineered_data.csv')
output_path = Path('../data/processed/robyn_input.csv')

# Expected columns for Robyn preparation
required_columns = [
    'Date',
    'Sales_Value',
    'TV_Impressions',
    'YouTube_Impressions',
    'Facebook_Impressions',
    'Instagram_Impressions',
    'Trade_Spend',
    'Feature_Flag',
    'Display_Flag',
    'TPR_Flag'
]

print(f'Loading dataset from: {input_path}')
df = pd.read_csv(input_path)
print(f'Raw shape: {df.shape}')

# Handle datasets where date is stored as Week
if 'Date' not in df.columns and 'Week' in df.columns:
    df = df.rename(columns={'Week': 'Date'})

# 1) Data validation
missing_required = [col for col in required_columns if col not in df.columns]
if missing_required:
    raise ValueError(f'Missing required columns: {missing_required}')

# Keep only Robyn-relevant columns
robyn_base = df[required_columns].copy()

# 2) Date formatting
robyn_base['Date'] = pd.to_datetime(robyn_base['Date'], errors='coerce')
invalid_dates = robyn_base['Date'].isna().sum()
if invalid_dates > 0:
    print(f'Warning: {invalid_dates} invalid date rows found and dropped.')
    robyn_base = robyn_base.dropna(subset=['Date'])

# 3) Null value checking (before fill)
null_report_before = robyn_base.isnull().sum()
print('\nNull values before treatment:')
print(null_report_before[null_report_before > 0] if (null_report_before > 0).any() else 'No nulls found')

# Numeric cleanup
numeric_columns = [
    'Sales_Value',
    'TV_Impressions',
    'YouTube_Impressions',
    'Facebook_Impressions',
    'Instagram_Impressions',
    'Trade_Spend',
    'Feature_Flag',
    'Display_Flag',
    'TPR_Flag'
]

for col in numeric_columns:
    robyn_base[col] = pd.to_numeric(robyn_base[col], errors='coerce')

# Fill null numeric values with 0 for robust aggregation
robyn_base[numeric_columns] = robyn_base[numeric_columns].fillna(0)

# Ensure flags are binary-like 0/1 before aggregation
for flag_col in ['Feature_Flag', 'Display_Flag', 'TPR_Flag']:
    robyn_base[flag_col] = (robyn_base[flag_col] > 0).astype(int)

# 4) Weekly aggregation
# Using week start (Monday) to create weekly buckets expected by MMM pipelines.
robyn_base['week_start'] = robyn_base['Date'] - pd.to_timedelta(robyn_base['Date'].dt.weekday, unit='D')

agg_map = {
    'Sales_Value': 'sum',
    'TV_Impressions': 'sum',
    'YouTube_Impressions': 'sum',
    'Facebook_Impressions': 'sum',
    'Instagram_Impressions': 'sum',
    'Trade_Spend': 'sum',
    'Feature_Flag': 'max',
    'Display_Flag': 'max',
    'TPR_Flag': 'max'
}

robyn_weekly = (
    robyn_base
    .groupby('week_start', as_index=False)
    .agg(agg_map)
    .sort_values('week_start')
    .reset_index(drop=True)
)

# 5) Robyn input dataset creation
robyn_input = robyn_weekly.rename(columns={'week_start': 'Date'})
robyn_input['Date'] = robyn_input['Date'].dt.strftime('%Y-%m-%d')

# Post-aggregation null check
null_report_after = robyn_input.isnull().sum()
print('\nNull values after weekly aggregation:')
print(null_report_after[null_report_after > 0] if (null_report_after > 0).any() else 'No nulls found')

# 6) Save robyn_input.csv
output_path.parent.mkdir(parents=True, exist_ok=True)
robyn_input.to_csv(output_path, index=False)

print(f'\nRobyn input created successfully: {output_path}')
print(f'Robyn input shape: {robyn_input.shape}')
print('\nPreview:')
print(robyn_input.head())

Loading dataset from: ..\data\processed\feature_engineered_data.csv
Raw shape: (11232, 62)

Null values before treatment:
No nulls found

Null values after weekly aggregation:
No nulls found

Robyn input created successfully: ..\data\processed\robyn_input.csv
Robyn input shape: (156, 10)

Preview:
         Date   Sales_Value  TV_Impressions  YouTube_Impressions  \
0  2022-07-04  1.010975e+06    5.021806e+07         2.482168e+07   
1  2022-07-11  1.330988e+06    4.894350e+07         2.352123e+07   
2  2022-07-18  1.563271e+06    4.864866e+07         2.177846e+07   
3  2022-07-25  1.315034e+06    4.976261e+07         2.492207e+07   
4  2022-08-01  1.344600e+06    5.135814e+07         2.438089e+07   

   Facebook_Impressions  Instagram_Impressions   Trade_Spend  Feature_Flag  \
0          1.350719e+07           6.315054e+06  1.026001e+06             0   
1          1.181995e+07           5.886242e+06  5.129495e+06             1   
2          1.336336e+07           5.692028e+06  6.188054e+